In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/DL_Project/data.zip"
extract_path = "/content/data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Done ✅")

Extraction Done ✅


In [ ]:
import os
import json
import shutil

base_path = "/content/data"
image_folder = os.path.join(base_path, "img")
train_json = os.path.join(base_path, "train.jsonl")

# Create new folders
output_train = "/content/dataset/train"
output_test = "/content/dataset/test"

for folder in [output_train+"/toxic", output_train+"/non_toxic",
               output_test+"/toxic", output_test+"/non_toxic"]:
    os.makedirs(folder, exist_ok=True)

# Read JSON and copy images
with open(train_json, 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    data = json.loads(line)
    img_name = data["img"]
    label = data["label"]

    src = os.path.join(image_folder, img_name.split("/")[-1])

    # 80% train, 20% test split
    if i < int(0.8 * len(lines)):
        if label == 1:
            dst = os.path.join(output_train, "toxic")
        else:
            dst = os.path.join(output_train, "non_toxic")
    else:
        if label == 1:
            dst = os.path.join(output_test, "toxic")
        else:
            dst = os.path.join(output_test, "non_toxic")

    if os.path.exists(src):
        shutil.copy(src, dst)

print("Dataset Organized Successfully ✅")

Dataset Organized Successfully ✅


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import EfficientNetB0
import matplotlib.pyplot as plt

# Reproducibility
tf.random.set_seed(42)

# Paths
train_dir = "/content/dataset/train"
test_dir = "/content/dataset/test"
MODEL_DIR = "/content/drive/MyDrive/DL_Projects/model"

# Data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.15,
    fill_mode="nearest",
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Training generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode="binary",
    subset="training",
    shuffle=True
)

# Validation generator
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode="binary",
    subset="validation",
    shuffle=False
)

# Test generator
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode="binary",
    shuffle=False
)


# EfficientNetB0 model
def create_improved_model(input_shape=(224, 224, 3)):
    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
        pooling="avg"
    )

    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.BatchNormalization(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.BatchNormalization(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ])

    return model, base_model


model, base_model = create_improved_model()

# Compile model
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()

# Training callbacks
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        f"{MODEL_DIR}/best_hateful_meme_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


# Phase 1: Train with frozen EfficientNet
print("Phase 1: Training with frozen base model...")

history_phase1 = model.fit(
    train_generator,
    epochs=15,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)


# Phase 2: Fine-tuning
print("\nPhase 2: Fine-tuning with unfrozen layers...")

base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

history_phase2 = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)


# Evaluation
print("\nEvaluating on test set...")

test_loss, test_acc, test_precision, test_recall, test_auc = model.evaluate(
    test_generator
)

f1_score = (
    2 * (test_precision * test_recall) /
    (test_precision + test_recall)
)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Test F1-Score: {f1_score:.4f}")


# Training history
def plot_history(history, phase_name=""):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    axes[0, 0].plot(history.history["accuracy"], label="Train Accuracy")
    axes[0, 0].plot(history.history["val_accuracy"], label="Val Accuracy")
    axes[0, 0].set_title(f"{phase_name} Accuracy")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Accuracy")
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    axes[0, 1].plot(history.history["loss"], label="Train Loss")
    axes[0, 1].plot(history.history["val_loss"], label="Val Loss")
    axes[0, 1].set_title(f"{phase_name} Loss")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Loss")
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    axes[1, 0].plot(history.history["precision"], label="Train Precision")
    axes[1, 0].plot(history.history["val_precision"], label="Val Precision")
    axes[1, 0].set_title(f"{phase_name} Precision")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Precision")
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    axes[1, 1].plot(history.history["recall"], label="Train Recall")
    axes[1, 1].plot(history.history["val_recall"], label="Val Recall")
    axes[1, 1].set_title(f"{phase_name} Recall")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Recall")
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.show()


plot_history(history_phase1, "Phase 1")
plot_history(history_phase2, "Phase 2")


# Save final model
final_model_path = f"{MODEL_DIR}/hateful_meme_final2.keras"
model.save(final_model_path)

print(f"\nFinal model saved successfully at: {final_model_path}")